# 13 - Recommendation RAG


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Understand recommendation + retrieval + generation as separate stages.
- Build a simple content-based recommendation baseline.
- Add RAG-style explanations using retrieved product facts.
- Learn why recommendations need constraints, not only similarity.

## Core AI concept

Recommendation RAG retrieves candidate items, then uses an LLM to explain or personalize the shortlist. Retrieval answers 'which items are plausible?'; generation answers 'why are these items suitable?' This is different from letting an LLM invent products.

## Architecture / mental model

```text
User preferences
 ↓
Candidate retrieval
 ↓
Ranking / filtering
 ↓
Evidence retrieval
 ↓
LLM explanation
 ↓
Recommendations + reasons
```

## Connection to the supplied learning material

This is a new application pattern built from the supplied retrieval and RAG concepts. The source notebooks support the retrieval/generation mechanics, while the recommendation logic is an applied extension.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

In [ ]:
%pip install -qU scikit-learn pandas

### Product catalog

In [ ]:
import pandas as pd

products = pd.DataFrame([
    {"id": "p1", "name": "Budget Laptop", "text": "portable laptop, coding, 8GB RAM, low price", "price": 600},
    {"id": "p2", "name": "Creator Laptop", "text": "laptop, video editing, 32GB RAM, strong GPU", "price": 1500},
    {"id": "p3", "name": "Student Laptop", "text": "lightweight laptop, battery life, study, coding", "price": 800},
    {"id": "p4", "name": "Gaming Laptop", "text": "gaming, strong GPU, high refresh rate", "price": 1400},
])

products

### Content-based retrieval

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(products["text"])

def recommend(query, budget=None, k=3):
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, matrix)[0]
    ranked = products.assign(score=scores).sort_values("score", ascending=False)
    if budget is not None:
        ranked = ranked[ranked["price"] <= budget]
    return ranked.head(k)

print(recommend("lightweight laptop for study and coding", budget=900))

### RAG-style explanation

In [ ]:
recommendations = recommend("lightweight laptop for study and coding", budget=900)

context = "\n".join(
    f"{row.id}: {row.name} | {row.text} | price={row.price}"
    for _, row in recommendations.iterrows()
)

print("Evidence for LLM:\n", context)
# Pass only these retrieved facts to an LLM and ask it to explain
# why each item matches the user's needs.

## Evaluation

Evaluate Precision@K, NDCG/ranking quality, catalog coverage, constraint violations, and explanation grounding. A recommendation should never cite features absent from the catalog evidence.

## Improvement ideas

Add metadata filters for budget/brand/specifications, hybrid ranking, user history, diversity, business rules, and feedback-based re-ranking.

## Practice exercises

1. Add a budget filter.
2. Add diversity so all recommendations are not nearly identical.
3. Create a grounded explanation prompt.

## Recommendation RAG vs normal RAG

Normal RAG asks:
> What evidence answers this question?

Recommendation RAG asks:
> Which items should be considered, and what evidence explains why each is a good fit?

This means the retrieval stage usually needs **candidate generation + constraints + ranking**, not just similarity search.

## Recommendation evaluation

Useful metrics depend on the product:
- Precision@K: useful recommendations in the top K
- Recall@K: did we surface relevant candidates?
- NDCG: ranking quality
- constraint satisfaction: did every result obey budget/category rules?
- diversity: are the results meaningfully different?
- grounding: are explanations supported by catalog facts?

The LLM should not be allowed to invent price/specification data.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create